# COLMAP Bundle Adjustment for RoomScanAlpha

Refines ARKit camera poses via COLMAP feature extraction, matching, and bundle adjustment.

**Runtime**: GPU (T4) — Runtime > Change runtime type

**Input**: Upload `scan_data.zip` from Desktop

**Output**: Downloads `refined_output.zip` with refined poses

## 1. Install pycolmap with CUDA

In [ ]:
!pip install pycolmap-cuda12

import pycolmap
print("pycolmap version:", pycolmap.__version__)
print("has_cuda:", pycolmap.has_cuda)

## 2. Upload scan data

In [ ]:
from google.colab import files
import os, zipfile

uploaded = files.upload()

os.makedirs('/content/work', exist_ok=True)
with zipfile.ZipFile('scan_data.zip', 'r') as z:
    z.extractall('/content/work')

print('Images:', len(os.listdir('/content/work/images')))
print('Sparse:', os.listdir('/content/work/sparse'))

## 3. Feature Extraction (GPU, ~1 min)

In [ ]:
import pycolmap
import os

os.chdir('/content/work')
if os.path.exists('database.db'):
    os.remove('database.db')

pycolmap.extract_features(
    database_path='database.db',
    image_path='images',
    camera_model='PINHOLE',
    camera_params=[1428.3756, 1428.3756, 958.9948, 725.38055],
    single_camera=True,
    device=pycolmap.Device.cuda if pycolmap.has_cuda else pycolmap.Device.cpu,
)
print('=== Feature extraction complete ===')

## 4. Exhaustive Matching (GPU, ~1-2 min)

In [ ]:
import pycolmap
import os

os.chdir('/content/work')

pycolmap.match_exhaustive(
    database_path='database.db',
    device=pycolmap.Device.cuda if pycolmap.has_cuda else pycolmap.Device.cpu,
)
print('=== Matching complete ===')

## 5. Triangulate Points with ARKit Poses (~1 min)

In [ ]:
import pycolmap
import os, shutil

os.chdir('/content/work')
if os.path.exists('sparse_triangulated'):
    shutil.rmtree('sparse_triangulated')
os.makedirs('sparse_triangulated')

pycolmap.triangulate_points(
    database_path='database.db',
    image_path='images',
    input_path='sparse',
    output_path='sparse_triangulated',
)
print('=== Triangulation complete ===')
print('Output:', os.listdir('sparse_triangulated'))

## 5b. Fallback: Full Mapper (if triangulation fails)

In [ ]:
# Only run this if Step 5 failed
import pycolmap
import os, shutil

os.chdir('/content/work')
if os.path.exists('sparse_auto'):
    shutil.rmtree('sparse_auto')
os.makedirs('sparse_auto')

maps = pycolmap.incremental_mapping(
    database_path='database.db',
    image_path='images',
    output_path='sparse_auto',
)
print(f'=== Mapper complete: {len(maps)} model(s) ===')
for i, rec in maps.items():
    print(f'  Model {i}: {rec.summary()}')

## 6. Bundle Adjustment (refine poses only)

In [ ]:
import pycolmap
import os, shutil

os.chdir('/content/work')

# Find the reconstruction
if os.path.exists('sparse_triangulated') and os.listdir('sparse_triangulated'):
    input_path = 'sparse_triangulated'
elif os.path.exists('sparse_auto/0'):
    input_path = 'sparse_auto/0'
else:
    raise RuntimeError('No reconstruction found. Run Step 5 or 5b first.')

print(f'Loading reconstruction from: {input_path}')
rec = pycolmap.Reconstruction(input_path)
print(f'Before BA: {rec.summary()}')

# Bundle adjustment — refine poses only, keep intrinsics fixed
ba_options = pycolmap.BundleAdjustmentOptions()
ba_options.refine_focal_length = False
ba_options.refine_principal_point = False
ba_options.refine_extra_params = False

pycolmap.bundle_adjustment(rec, ba_options)
print(f'After BA: {rec.summary()}')

# Save refined
if os.path.exists('sparse_refined'):
    shutil.rmtree('sparse_refined')
os.makedirs('sparse_refined')
rec.write(output_path='sparse_refined')

# Also save as text for easy inspection
if os.path.exists('sparse_refined_txt'):
    shutil.rmtree('sparse_refined_txt')
os.makedirs('sparse_refined_txt')
rec.write_text('sparse_refined_txt')

print('=== Bundle adjustment complete ===')
print('Refined:', os.listdir('sparse_refined_txt'))

## 7. Compare pose corrections

In [ ]:
import numpy as np

def parse_images_txt(path):
    poses = {}
    with open(path) as f:
        for line in f:
            if line.startswith('#') or line.strip() == '':
                continue
            parts = line.strip().split()
            if len(parts) >= 10:
                name = parts[9]
                qw,qx,qy,qz = float(parts[1]),float(parts[2]),float(parts[3]),float(parts[4])
                tx,ty,tz = float(parts[5]),float(parts[6]),float(parts[7])
                poses[name] = (qw,qx,qy,qz,tx,ty,tz)
    return poses

def quat_to_rotmat(qw,qx,qy,qz):
    return np.array([
        [1-2*(qy*qy+qz*qz), 2*(qx*qy-qw*qz), 2*(qx*qz+qw*qy)],
        [2*(qx*qy+qw*qz), 1-2*(qx*qx+qz*qz), 2*(qy*qz-qw*qx)],
        [2*(qx*qz-qw*qy), 2*(qy*qz+qw*qx), 1-2*(qx*qx+qy*qy)]
    ])

def camera_position(qw,qx,qy,qz,tx,ty,tz):
    R = quat_to_rotmat(qw,qx,qy,qz)
    return -R.T @ np.array([tx,ty,tz])

original = parse_images_txt('/content/work/sparse/images.txt')
refined = parse_images_txt('/content/work/sparse_refined_txt/images.txt')

common = sorted(set(original) & set(refined))
print(f'Original: {len(original)} | Refined: {len(refined)} | Common: {len(common)}')

diffs = []
for name in common:
    pos_orig = camera_position(*original[name])
    pos_ref = camera_position(*refined[name])
    diffs.append(np.linalg.norm(pos_orig - pos_ref))

diffs = np.array(diffs)
print(f'\nPose corrections (meters):')
print(f'  Mean: {diffs.mean():.4f}m ({diffs.mean()*100:.1f}cm)')
print(f'  Max:  {diffs.max():.4f}m ({diffs.max()*100:.1f}cm)')
print(f'  Std:  {diffs.std():.4f}m')
print(f'\nExpected: 1-3cm if BA is working. >10cm = problem.')

## 8. Download refined poses

In [ ]:
import zipfile
from google.colab import files

with zipfile.ZipFile('/content/refined_output.zip', 'w', zipfile.ZIP_DEFLATED) as zf:
    for f in ['cameras.txt', 'images.txt', 'points3D.txt']:
        path = f'/content/work/sparse_refined_txt/{f}'
        if os.path.exists(path):
            zf.write(path, f'sparse_refined/{f}')

print('Downloading refined_output.zip...')
files.download('/content/refined_output.zip')